# Geocoding Completion Rate by State
Produces a tiered completion rate report and a US choropleth map.

Input: output/1_derived/final_truck_stops.csv
Output:
  - output/2_analysis/tables/completion_rate_by_state.txt
  - output/2_analysis/figures/completion_rate_by_state_map.html
  - output/2_analysis/figures/completion_rate_by_state_map.png

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px


def find_project_root(start=None, markers=('run_all.py', 'requirements.txt')):
    p = Path(start or Path.cwd()).resolve()
    for candidate in (p, *p.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise FileNotFoundError('Project root not found')


PROJECT_ROOT = find_project_root()
IN_FILE = PROJECT_ROOT / 'output' / '1_derived' / 'final_truck_stops.csv'
TABLES_DIR = PROJECT_ROOT / 'output' / '2_analysis' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'output' / '2_analysis' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded {len(df):,} rows')

In [ ]:
# Compute completion rate by state
has_coords = df['Final_Lat'].notna() & df['Final_Long'].notna()

by_state = df.groupby('state').agg(
    total=('state', 'size'),
    complete=('Final_Lat', lambda x: x.notna().sum())
).reset_index()
by_state['pct'] = (by_state['complete'] / by_state['total'] * 100).round(2)
by_state = by_state.sort_values('pct', ascending=False).reset_index(drop=True)

# Assign tiers
def tier(pct):
    if pct >= 90:
        return 'EXCELLENT (\u226590% completion)'
    elif pct >= 70:
        return 'GOOD (70-89% completion)'
    else:
        return 'NEEDS IMPROVEMENT (<70% completion)'

by_state['tier'] = by_state['pct'].apply(tier)

# Build the text report
lines = ['Completion Rate Raw:', '']
for tier_name, group in by_state.groupby('tier', sort=False):
    # Sort tiers in the right order
    pass

# Re-sort to get tiers in order: EXCELLENT, GOOD, NEEDS IMPROVEMENT
tier_order = [
    'EXCELLENT (\u226590% completion)',
    'GOOD (70-89% completion)',
    'NEEDS IMPROVEMENT (<70% completion)'
]

for t in tier_order:
    group = by_state[by_state['tier'] == t].sort_values('pct', ascending=False)
    if group.empty:
        continue
    lines.append(f'{t}: {len(group)} states')
    lines.append('-' * 80)
    for _, row in group.iterrows():
        lines.append(f"  {row['state']:<16s} {row['pct']:>6.2f}%   ({row['complete']:,}/{row['total']:,} truck stops)")

report = '\n'.join(lines)
print(report)

# Save
out_txt = TABLES_DIR / 'completion_rate_by_state.txt'
out_txt.write_text(report, encoding='utf-8')
print(f'\nSaved: {out_txt}')

In [ ]:
# Choropleth map
fig = px.choropleth(
    by_state,
    locations='state',
    locationmode='USA-states',
    color='pct',
    color_continuous_scale=[[0, '#d73027'], [0.5, '#fee08b'], [1, '#1a9850']],
    range_color=[by_state['pct'].min() - 1, by_state['pct'].max() + 1],
    scope='usa',
    labels={'pct': 'Completion<br>Rate (%)'},
    hover_data={'state': True, 'pct': ':.2f', 'complete': ':,', 'total': ':,'},
    title='Geocoding Completion Rate by State<br><sup>Darker green = higher completion rate</sup>'
)
fig.update_layout(
    geo=dict(bgcolor='rgba(0,0,0,0)'),
    paper_bgcolor='white',
    margin=dict(l=0, r=0, t=60, b=0),
    width=900,
    height=500
)

# Save HTML (interactive)
html_path = FIGURES_DIR / 'completion_rate_by_state_map.html'
fig.write_html(str(html_path))
print(f'Saved: {html_path}')

# Save PNG (static)
try:
    png_path = FIGURES_DIR / 'completion_rate_by_state_map.png'
    fig.write_image(str(png_path), scale=2)
    print(f'Saved: {png_path}')
except Exception as e:
    print(f'PNG export skipped ({e}). Install kaleido: pip install kaleido')

fig.show()